# Inference: Generating Videos

In this notebook we use a trained NanoDiT checkpoint to generate videos from scratch. We will explore:

1. The full denoising loop and how noise gradually becomes structured video
2. Classifier-Free Guidance (CFG) -- the technique that lets you control how strongly the model follows the text prompt
3. The effect of the number of denoising steps on generation quality
4. Saving generated videos as GIF files

## The Sampling Process

Generating a video with a diffusion model is the reverse of the training process:

1. **Start from pure noise**: $x_T \sim \mathcal{N}(0, I)$ in latent space
2. **Iteratively denoise**: At each step $t$, the model predicts the velocity $\hat{v} = \text{DiT}(x_t, t, \text{context})$
3. **Euler step**: Update the sample: $x_{t-1} = x_t + \hat{v} \cdot (\sigma_{t-1} - \sigma_t)$
4. **Decode**: Pass the final clean latent $x_0$ through the VAE decoder to get pixel-space video

The noise schedule is controlled by the **shift** parameter. With `shift=5.0` (Wan's default), the model spends more compute on the high-noise steps where global structure is being formed, and fewer steps on the low-noise fine-detail steps.

```
x_T (pure noise) --> [denoise step N] --> ... --> [denoise step 1] --> x_0 (clean latent)
                                                                          |
                                                                     [VAE decode]
                                                                          |
                                                                     video pixels
```

In [ ]:
import sys
import os

# Add the project root to Python path so we can import nano_video_gen
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from nano_video_gen.model.nano_dit import NanoDiT
from nano_video_gen.model.nano_vae import DummyVAE
from nano_video_gen.diffusion.flow_match import FlowMatchScheduler
from nano_video_gen.data.dataset import SimpleTextEncoder
from nano_video_gen.visualization.viz import (
    visualize_denoising_process,
    save_video_grid,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load the Trained Checkpoint

We load the checkpoint saved by the training notebook (`06_training.ipynb`). The checkpoint contains the state dicts for the DiT model, the VAE, and the text encoder, along with metadata like the number of unique prompts.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Path to the trained checkpoint
checkpoint_path = os.path.join(os.getcwd(), '..', 'outputs', 'checkpoint_final.pt')

if not os.path.exists(checkpoint_path):
    print(f"\nERROR: Checkpoint not found at: {checkpoint_path}")
    print("\nPlease run notebook 06_training.ipynb first to train the model.")
    print("Alternatively, if you saved the checkpoint elsewhere, update the path above.")
    raise FileNotFoundError(
        f"No checkpoint found at {checkpoint_path}. "
        "Run 06_training.ipynb first."
    )

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Final training loss: {checkpoint['losses'][-1]:.4f}")

# Get num_prompts from checkpoint (or default)
num_prompts = checkpoint.get('num_prompts', 12)

In [ ]:
# Initialize models with the same architecture used during training
model = NanoDiT(
    dim=128,
    in_dim=4,
    ffn_dim=512,
    out_dim=4,
    text_dim=64,
    freq_dim=64,
    num_heads=4,
    num_layers=2,
    patch_size=(1, 2, 2),
).to(device)

vae = DummyVAE(
    in_channels=3,
    latent_channels=4,
    spatial_factor=4,
    temporal_factor=4,
).to(device)

text_encoder = SimpleTextEncoder(
    num_prompts=num_prompts,
    text_dim=64,
    seq_len=8,
).to(device)

# Load state dicts
model.load_state_dict(checkpoint['model'])
vae.load_state_dict(checkpoint['vae'])
text_encoder.load_state_dict(checkpoint['text_encoder'])

# Set all models to eval mode
model.eval()
vae.eval()
text_encoder.eval()

print("All models loaded and set to eval mode.")

## 2. The Denoising Loop

Let's walk through the denoising process step by step and collect intermediate results so we can visualize how the video emerges from noise.

At each step, the scheduler provides the current noise level $\sigma_t$. The model sees the noisy sample $x_t$ and predicts the velocity $\hat{v} = \epsilon - x_0$. The Euler update then takes a step toward the clean data:

$$x_{t-1} = x_t + \hat{v} \cdot (\sigma_{t-1} - \sigma_t)$$

Since $\sigma_{t-1} < \sigma_t$ (noise level decreases), the factor $(\sigma_{t-1} - \sigma_t)$ is negative, so the update effectively subtracts the predicted noise direction, gradually revealing the clean signal.

In [ ]:
# Full inference with denoising visualization
scheduler = FlowMatchScheduler()
scheduler.set_timesteps(50, shift=5.0)

# Start from pure noise in latent space
x = torch.randn(1, 4, 4, 16, 16, device=device)

# Condition on the first prompt (prompt index 0)
ctx = text_encoder(torch.tensor([0], device=device))

# Collect intermediate decoded frames for visualization
denoising_frames = []

with torch.no_grad():
    for i, t in enumerate(tqdm(scheduler.timesteps, desc="Denoising (50 steps)")):
        v_pred = model(x, t.unsqueeze(0).to(device), ctx)
        x = scheduler.step(v_pred, t, x)

        # Save decoded frames at regular intervals
        if i % 5 == 0:
            decoded = vae.decode(x).cpu()
            denoising_frames.append(decoded)

    # Always include the final result
    final_decoded = vae.decode(x).cpu()
    denoising_frames.append(final_decoded)

print(f"Collected {len(denoising_frames)} snapshots of the denoising process.")
print(f"Final video shape: {final_decoded.shape}")  # [1, 3, 16, 64, 64]

In [ ]:
# Visualize the denoising process
# This shows how the first temporal frame evolves from noise to a structured image
fig = visualize_denoising_process(denoising_frames, frame_idx=0, figsize=(18, 3))
plt.show()

# Also show the middle temporal frame
fig = visualize_denoising_process(denoising_frames, frame_idx=8, figsize=(18, 3))
plt.show()

## 3. Classifier-Free Guidance (CFG)

**Classifier-Free Guidance** (Ho & Salimans, 2022) is a key technique used in all modern diffusion models (Stable Diffusion, DALL-E, Wan, etc.) to improve the alignment between generated outputs and the conditioning signal (e.g., text prompt).

The idea is simple:

1. Run the model **with** the text conditioning to get $\hat{v}_{\text{cond}}$
2. Run the model **without** conditioning (using a null/zero context) to get $\hat{v}_{\text{uncond}}$
3. **Interpolate** between them with a guidance scale $w$:

$$\hat{v}_{\text{guided}} = \hat{v}_{\text{uncond}} + w \cdot (\hat{v}_{\text{cond}} - \hat{v}_{\text{uncond}})$$

- When $w = 1.0$: No guidance (standard conditional generation)
- When $w > 1.0$: The model amplifies the difference between conditioned and unconditioned predictions, producing outputs more strongly aligned with the prompt
- Wan 2.1 default: $w = 5.0$

The tradeoff: higher guidance scales produce sharper, more prompt-faithful results but can reduce diversity and introduce artifacts if pushed too high.

**Note**: Our SimpleTextEncoder uses a learned embedding table. To simulate "no conditioning," we can use a zero-vector context or a special null prompt index. Below is a demonstration of the concept.

In [ ]:
# Demonstrate CFG: compare conditioned vs unconditioned predictions
scheduler_cfg = FlowMatchScheduler()
scheduler_cfg.set_timesteps(30, shift=5.0)

# Create a noisy sample at a mid-level noise
x_start = torch.randn(1, 4, 4, 16, 16, device=device)

# Conditioned context (prompt index 0)
ctx_cond = text_encoder(torch.tensor([0], device=device))

# Unconditioned context (zero vector -- simulating null prompt)
ctx_uncond = torch.zeros_like(ctx_cond)

# Pick a mid-level timestep to illustrate the difference
t_mid = scheduler_cfg.timesteps[len(scheduler_cfg.timesteps) // 2]

with torch.no_grad():
    v_cond = model(x_start, t_mid.unsqueeze(0).to(device), ctx_cond)
    v_uncond = model(x_start, t_mid.unsqueeze(0).to(device), ctx_uncond)

# Show the difference between conditioned and unconditioned predictions
diff = (v_cond - v_uncond).abs()

fig, axes = plt.subplots(1, 3, figsize=(14, 3))

# Show channel 0 of the first temporal frame for each
for ax, tensor, title in zip(
    axes,
    [v_cond, v_uncond, diff],
    ['v_cond (with text)', 'v_uncond (no text)', '|v_cond - v_uncond|']
):
    img = tensor[0, 0, 0].detach().cpu().numpy()  # [H_lat, W_lat]
    im = ax.imshow(img, cmap='coolwarm', aspect='equal')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f'Classifier-Free Guidance: Conditional vs Unconditional Velocity (t={t_mid.item():.0f})',
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(f"Mean |v_cond - v_uncond|: {diff.mean().item():.4f}")
print(f"This difference is what CFG amplifies by the guidance scale w.")
print(f"With w=5.0 (Wan default): v_guided = v_uncond + 5.0 * (v_cond - v_uncond)")

In [ ]:
# Generate with different CFG scales for comparison
cfg_scales = [1.0, 3.0, 5.0]
cfg_results = []

# Use a fixed seed for fair comparison
seed = 42

for w in cfg_scales:
    torch.manual_seed(seed)
    x = torch.randn(1, 4, 4, 16, 16, device=device)

    scheduler_cfg.set_timesteps(30, shift=5.0)

    with torch.no_grad():
        for t in scheduler_cfg.timesteps:
            t_batch = t.unsqueeze(0).to(device)
            v_cond = model(x, t_batch, ctx_cond)
            v_uncond = model(x, t_batch, ctx_uncond)
            # Apply CFG
            v_guided = v_uncond + w * (v_cond - v_uncond)
            x = scheduler_cfg.step(v_guided, t, x)

        video = vae.decode(x).cpu()
    cfg_results.append(video)

# Display frame 0 from each CFG scale
fig, axes = plt.subplots(1, len(cfg_scales), figsize=(4 * len(cfg_scales), 4))
for i, (w, video) in enumerate(zip(cfg_scales, cfg_results)):
    frame = video[0, :, 0]  # [3, H, W]
    frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(frame)
    axes[i].set_title(f'CFG scale = {w}')
    axes[i].axis('off')

fig.suptitle('Effect of Classifier-Free Guidance Scale', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Effect of Number of Steps

The number of denoising steps controls the tradeoff between speed and quality:

- **Fewer steps** (e.g., 10): Faster generation, but each step must make larger jumps, leading to coarser results.
- **More steps** (e.g., 50): Slower generation, but smaller, more precise denoising steps yield better quality.

Wan 2.1 typically uses 50 steps for high-quality generation. Let's compare 10, 20, and 50 steps.

In [ ]:
step_counts = [10, 20, 50]
step_results = []

# Use the same starting noise for fair comparison
seed = 123
ctx = text_encoder(torch.tensor([0], device=device))

for num_steps in step_counts:
    torch.manual_seed(seed)
    x = torch.randn(1, 4, 4, 16, 16, device=device)

    sched = FlowMatchScheduler()
    sched.set_timesteps(num_steps, shift=5.0)

    with torch.no_grad():
        for t in sched.timesteps:
            v_pred = model(x, t.unsqueeze(0).to(device), ctx)
            x = sched.step(v_pred, t, x)

        video = vae.decode(x).cpu()
    step_results.append(video)
    print(f"  {num_steps:2d} steps done.")

# Display first and last frames for each step count
fig, axes = plt.subplots(2, len(step_counts), figsize=(4 * len(step_counts), 7))

for col, (n, video) in enumerate(zip(step_counts, step_results)):
    for row, (frame_idx, frame_label) in enumerate([(0, 'First frame'), (-1, 'Last frame')]):
        frame = video[0, :, frame_idx]  # [3, H, W]
        frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row, col].imshow(frame)
        axes[row, col].set_title(f'{n} steps - {frame_label}')
        axes[row, col].axis('off')

fig.suptitle('Effect of Denoising Step Count on Generation Quality', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Save Generated Videos as GIFs

Let's generate a batch of videos with different prompt indices and save them as GIF files using the `save_video_grid` utility.

In [ ]:
# Generate videos for multiple prompts
num_to_generate = min(num_prompts, 4)  # Generate up to 4 videos
generated_videos = []

sched = FlowMatchScheduler()
sched.set_timesteps(30, shift=5.0)

for prompt_idx in range(num_to_generate):
    torch.manual_seed(prompt_idx * 7 + 42)  # Different seed per prompt
    x = torch.randn(1, 4, 4, 16, 16, device=device)
    ctx = text_encoder(torch.tensor([prompt_idx], device=device))

    with torch.no_grad():
        for t in sched.timesteps:
            v_pred = model(x, t.unsqueeze(0).to(device), ctx)
            x = sched.step(v_pred, t, x)
        video = vae.decode(x).cpu()

    generated_videos.append(video[0])  # [3, T, H, W]
    print(f"  Generated video for prompt {prompt_idx}")

# Save as a GIF grid
output_dir = os.path.join(os.getcwd(), '..', 'outputs')
os.makedirs(output_dir, exist_ok=True)
gif_path = os.path.join(output_dir, 'generated_videos.gif')

save_video_grid(generated_videos, gif_path, nrow=2, fps=8)
print(f"\nSaved video grid to: {gif_path}")

In [ ]:
# Display a summary grid: one frame from each generated video
fig, axes = plt.subplots(1, num_to_generate, figsize=(4 * num_to_generate, 4))
if num_to_generate == 1:
    axes = [axes]

for i, video in enumerate(generated_videos):
    # Show the middle frame
    mid = video.shape[1] // 2
    frame = video[:, mid]  # [3, H, W]
    frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(frame)
    axes[i].set_title(f'Prompt {i} (frame {mid})')
    axes[i].axis('off')

fig.suptitle('Generated Videos -- Middle Frame', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

In this notebook we covered the complete inference pipeline for video generation with a flow matching diffusion model:

1. **Denoising loop**: Starting from Gaussian noise in latent space, we iteratively applied the trained DiT model to predict the velocity and used Euler steps to move toward a clean sample.

2. **Classifier-Free Guidance (CFG)**: By comparing conditioned and unconditioned model predictions, we can amplify prompt-following behavior. The formula $\hat{v} = \hat{v}_{\text{uncond}} + w \cdot (\hat{v}_{\text{cond}} - \hat{v}_{\text{uncond}})$ with $w > 1$ pushes generation toward the conditioning signal.

3. **Step count tradeoff**: More denoising steps generally yield better quality at the cost of more compute. For this nano model, 20--50 steps work well.

4. **Visualization**: We tracked intermediate denoising states to see how global structure forms first (high-noise steps) and fine details emerge later (low-noise steps).

### Where to go from here

To scale this approach to production-quality video generation, you would:

| What to scale | Nano (this tutorial) | Production (Wan 2.1) |
|--------------|---------------------|----------------------|
| Model dim | 128 | 5120 |
| Transformer layers | 2 | 40 |
| Parameters | ~1.5M | ~14B |
| VAE | DummyVAE (Conv3d) | CausalConv3d + Attention |
| Text encoder | Embedding table | T5 (4096-dim, ~4B params) |
| Training data | 60 synthetic clips | Millions of real videos |
| Training compute | Minutes on 1 GPU | Weeks on hundreds of GPUs |
| Resolution | 64x64, 16 frames | 720p+, 81+ frames |

The architecture and training procedure are *exactly the same* -- the only difference is scale.